# Training LoRA Qwen untuk KKA — Cloud GPU (Jupyter)

Notebook ini untuk instance GPU sewaan dengan template **PyTorch + Jupyter** (Ubuntu),
misalnya RTX 5090 / 4090 / 3090 / A100. Untuk Google Colab tetap pakai
`Training_LoRA_Colab.ipynb`.

Perbedaan dengan versi Colab:
- Tidak memakai Google Drive / `google.colab`; semua file ada di disk instance.
- Ada pemeriksaan arsitektur GPU (RTX 50xx/Blackwell butuh PyTorch build CUDA ≥ 12.8).
- Mendukung `--precision bf16`, `--batch-size`, dan `--grad-accum` pada `training/train_lora.py`.

> Disk instance **hilang saat instance dihapus**. Unduh adaptor final (langkah 7) sebelum masa sewa habis.

Jalankan sel secara berurutan.

## 1. Masukkan proyek ke instance

Pilih salah satu, lewat **Terminal** Jupyter:

**A. Git (disarankan)**
```bash
git clone https://github.com/<username>/kka_lab.git
```

**B. ZIP** — di Windows, kompres folder `kka_lab` **tanpa** `.venv` (venv Windows tidak jalan di Linux),
lalu drag-and-drop `kka_lab.zip` ke panel file Jupyter. Sel berikut mengekstraknya otomatis.

Jangan unggah `.env` berisi token Astra; training tidak membutuhkannya.

In [ ]:
import os
import zipfile
from pathlib import Path

# Sesuaikan jika proyek diletakkan di folder lain.
PROJECT = Path('kka_lab').resolve()

zip_path = Path('kka_lab.zip')
if not PROJECT.exists() and zip_path.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('.')
    print('ZIP diekstrak.')

assert (PROJECT / 'training/train_lora.py').exists(), f'Proyek tidak ditemukan di {PROJECT}'
assert not (PROJECT / '.venv/Scripts').exists(), 'Hapus folder .venv Windows dari proyek ini.'
os.chdir(PROJECT)
print('Folder kerja:', Path.cwd())

## 2. Periksa GPU dan PyTorch

**Jangan menginstal ulang torch** dari README (`cu126`) — build itu untuk laptop RTX 3050 dan
tidak memiliki kernel untuk RTX 50xx (sm_120). Pakai torch bawaan template.
Sel ini berhenti jika build torch tidak mendukung GPU yang terpasang.

In [ ]:
import subprocess
import torch

assert torch.cuda.is_available(), 'GPU belum terdeteksi. Pilih template PyTorch dengan GPU.'
major, minor = torch.cuda.get_device_capability(0)
arch = f'sm_{major}{minor}'
arch_list = torch.cuda.get_arch_list()
props = torch.cuda.get_device_properties(0)

print('GPU          :', torch.cuda.get_device_name(0))
print('VRAM GiB     :', round(props.total_memory / 2**30, 1))
print('Compute cap. :', arch)
print('PyTorch      :', torch.__version__, '| CUDA build:', torch.version.cuda)
print('Arsitektur   :', arch_list)
print('bf16 support :', torch.cuda.is_bf16_supported())

if major >= 12:  # Blackwell konsumen (RTX 50xx)
    cuda_build = tuple(int(x) for x in (torch.version.cuda or '0.0').split('.')[:2])
    assert cuda_build >= (12, 8), (
        f'{arch} butuh PyTorch build CUDA >= 12.8, terpasang {torch.version.cuda}. '
        'Ganti template ke PyTorch terbaru (cu128 atau lebih baru).'
    )
assert arch in arch_list or f'compute_{major}{minor}' in arch_list, (
    f'Build torch ini tidak dikompilasi untuk {arch}. Ganti template PyTorch yang lebih baru.'
)

# Uji kernel GPU sungguhan (gagal di sini = build torch tidak cocok).
x = torch.randn(256, 256, device='cuda', dtype=torch.bfloat16)
torch.cuda.synchronize()
print('Uji matmul   :', float((x @ x).float().abs().mean()))
TORCH_BEFORE = torch.__version__
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

## 3. Instal dependensi proyek (Core + TRL DPO)

`requirements-core.txt` dan `requirements-training.txt` tidak memuat torch, sehingga torch bawaan template RTX 5090 (CUDA ≥ 12.8) tetap dipertahankan.
Sel ini memeriksa ulang bahwa versi torch tidak berubah.

In [ ]:
%pip install -q -r requirements-core.txt -r requirements-training.txt

In [ ]:
import importlib.metadata as md_

torch_after = md_.version('torch')
print('torch sebelum:', TORCH_BEFORE, '| sesudah:', torch_after)
assert torch_after.split('+')[0] == TORCH_BEFORE.split('+')[0], (
    'pip mengganti versi torch. Instal ulang torch bawaan template, lalu restart kernel.'
)
for pkg in ['transformers', 'peft', 'accelerate', 'trl', 'datasets']:
    print(f'{pkg:13}:', md_.version(pkg))
print('\nJika ada paket yang baru di-upgrade, pilih Kernel > Restart Kernel, '
      'lalu jalankan sel 1 dan 2 lagi sebelum lanjut.')

## 4. Unduh base model `Qwen/Qwen3-4B-Instruct-2507` & audit dataset

Di **RTX 5090 (32 GB VRAM)** kita melatih memakai **Native BF16 rsLoRA** (`--preset rtx5090-32gb`) tanpa kuantisasi 4-bit agar presisi training 100% cocok dengan inferensi BF16.
Sel ini mengunci `model_record.json` dan menjalankan audit kualitas dataset (`audit_dataset_quality.py --strict`) sebelum training dimulai.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

MODEL_ID = 'Qwen/Qwen3-4B-Instruct-2507'
RECORD = Path('model_record.json')
TRAIN_DATA = Path('data/lora/lora_train.jsonl')
VAL_DATA = Path('data/lora/lora_val.jsonl')
DPO_DATA = Path('data/lora/dpo_preferences.jsonl')

assert TRAIN_DATA.exists() and VAL_DATA.exists(), 'Dataset split data/lora/ belum lengkap.'
assert DPO_DATA.exists(), 'Dataset DPO data/lora/dpo_preferences.jsonl belum ada.'

subprocess.run([sys.executable, 'scripts/download_model.py', '--model', MODEL_ID, '--record', str(RECORD), '--force'], check=True)
subprocess.run([sys.executable, 'scripts/audit_dataset_quality.py', '--strict'], check=True)
subprocess.run([sys.executable, 'training/train_dpo.py', '--validate-only', '--data', str(DPO_DATA), '--record', str(RECORD), '--out', 'adapters/_validate'], check=True)
print('Record model:', json.loads(RECORD.read_text(encoding='utf-8')))

## 5. Tahap 1 — Latih SFT Native BF16 rsLoRA (`--preset rtx5090-32gb`)

Preset `rtx5090-32gb` otomatis menyetel Native BF16 (`--no-4bit`), `rsLoRA`, `adamw_torch_fused`, `TF32`, `group_by_length`, dan `NEFTune=5.0`, serta menjaga `pad_token` bawaan Qwen (`<|endoftext|>`).

In [ ]:
import shlex
import subprocess
from datetime import datetime

RUN_NAME = datetime.now().strftime('rtx5090_v4_%Y%m%d_%H%M')
SFT_OUT = Path('adapters') / f'{RUN_NAME}_sft'
DPO_OUT = Path('adapters') / f'{RUN_NAME}_dpo'
OUT = DPO_OUT
RESUME = ''

sft_cmd = [
    sys.executable, 'training/train_lora.py',
    '--data', str(TRAIN_DATA), '--val-data', str(VAL_DATA),
    '--record', str(RECORD), '--out', str(SFT_OUT),
    '--preset', 'rtx5090-32gb',
    '--epochs', '4', '--eval-strategy', 'epoch', '--early-stopping-patience', '2',
]
if RESUME:
    sft_cmd += ['--resume', RESUME]

print('Perintah SFT untuk Terminal/tmux:\n')
print('cd', shlex.quote(str(Path.cwd())), '&&', shlex.join(['python', *sft_cmd[1:]]), '\n')

proc = subprocess.Popen(sft_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
assert proc.returncode == 0, f'Training SFT gagal (exit code {proc.returncode}).'
print('\nSFT Selesai:', SFT_OUT)

## 6. Tahap 2 — Robust DPO Alignment (Anti-Halusinasi & Anti-Bias) + Evaluasi Ganda

Melanjutkan adaptor SFT dengan `trl.DPOTrainer` (`loss_type='robust'`, `beta=0.1`, `lr=5e-6`) sehingga tetap menghasilkan **1 adaptor LoRA drop-in** lalu langsung dievaluasi pada 40 kasus rubrik pedagogis dan 10 kasus anti-bias/strict-grounding.

In [ ]:
dpo_cmd = [
    sys.executable, 'training/train_dpo.py',
    '--data', str(DPO_DATA), '--record', str(RECORD),
    '--sft-adapter', str(SFT_OUT / 'final'),
    '--out', str(DPO_OUT),
    '--epochs', '2', '--lr', '5e-6', '--beta', '0.1',
    '--loss-type', 'robust', '--label-smoothing', '0.05',
]
print('Perintah DPO:\n', shlex.join(['python', *dpo_cmd[1:]]), '\n')
proc = subprocess.Popen(dpo_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
assert proc.returncode == 0, f'Training DPO gagal (exit code {proc.returncode}).'

subprocess.run([
    sys.executable, 'scripts/evaluate_assistant.py',
    '--local-adapter', str(DPO_OUT / 'final'),
    '--save-report', f'exports/{RUN_NAME}_eval_40.json',
    '--save-responses', f'exports/{RUN_NAME}_eval_40_responses.jsonl',
], check=True)
subprocess.run([
    sys.executable, 'scripts/evaluate_assistant.py',
    '--cases', 'evals/eval_bias_and_grounding.jsonl',
    '--local-adapter', str(DPO_OUT / 'final'),
    '--save-report', f'exports/{RUN_NAME}_eval_bias_grounding.json',
], check=True)
print('\nSelesai DPO + Evaluasi:', DPO_OUT)

### Jika training terputus

Isi `RESUME` dengan checkpoint yang benar-benar ada, misalnya
`adapters/run_YYYYMMDD_HHMM/checkpoints/checkpoint-N`, set `RUN_NAME` ke nama run yang sama,
lalu jalankan ulang sel di atas. Jangan resume jika data, model, atau konfigurasi telah diubah.

In [ ]:
for ckpt in sorted(Path('adapters').glob('*/checkpoints/checkpoint-*')):
    print(ckpt)

## 7. Periksa hasil dan kemas untuk diunduh

Sel ini membuat dua ZIP di folder `exports/`:
- `<run>-qwen-kka-lora.zip` — adaptor final (puluhan MB). Isinya folder `qwen-kka-lora/`.
- `<run>-records.zip` — catatan penelitian (`run_manifest`, `packages.lock`, `evaluation`,
  `trainer_state`). Checkpoint tidak ikut karena besar.

Unduh keduanya: klik kanan file di panel Jupyter → **Download**.

In [ ]:
import shutil

final = OUT / 'final'
assert (final / 'adapter_config.json').exists(), 'Adaptor final belum ada. Training belum selesai?'

evaluation = json.loads((OUT / 'evaluation.json').read_text(encoding='utf-8'))
manifest = json.loads((OUT / 'run_manifest.json').read_text(encoding='utf-8'))
print('GPU          :', manifest['gpu'])
print('Train/val    :', manifest['train_rows'], '/', manifest['validation_rows'])
print('Evaluasi     :', evaluation)

exports = Path('exports')
stage = exports / f'{RUN_NAME}_stage'
shutil.rmtree(stage, ignore_errors=True)
shutil.copytree(final, stage / 'qwen-kka-lora')
adapter_zip = shutil.make_archive(str(exports / f'{RUN_NAME}-qwen-kka-lora'), 'zip', stage)

records = exports / f'{RUN_NAME}_records'
shutil.rmtree(records, ignore_errors=True)
records.mkdir(parents=True)
for name in ['run_manifest.json', 'packages.lock.txt', 'evaluation.json', 'trainer_state.json']:
    if (OUT / name).exists():
        shutil.copy2(OUT / name, records / name)
records_zip = shutil.make_archive(str(exports / f'{RUN_NAME}-records'), 'zip', records)

for p in [adapter_zip, records_zip]:
    print(f'{p}  ({Path(p).stat().st_size / 2**20:.1f} MB)')

### Opsional: simpan ke Hugging Face Hub (private)

Cadangan selain unduhan manual. Butuh token **write** dari huggingface.co/settings/tokens.
Jangan menulis token langsung di notebook.

In [ ]:
# from huggingface_hub import login, HfApi
# login()  # tempel token saat diminta
# api = HfApi()
# repo_id = '<username>/qwen-kka-lora'
# api.create_repo(repo_id, private=True, exist_ok=True)
# api.upload_folder(folder_path=str(final), repo_id=repo_id, commit_message=RUN_NAME)

## 8. Pakai adaptor di laptop / server inferensi

1. Ekstrak `<run>-qwen-kka-lora.zip` ke folder `adapters/` proyek sehingga ada
   `adapters/qwen-kka-lora/adapter_config.json`.
2. Jalankan:
   ```bash
   python chat.py --mode terstruktur --adapter adapters/qwen-kka-lora
   python chat.py --mode umum --adapter adapters/qwen-kka-lora
   ```
   atau untuk API FastAPI, isi `MODEL_ADAPTER=adapters/qwen-kka-lora` di `.env`.

Adaptor tidak menggantikan model dasar; model dasar dan revisi pada `model_record.json`
harus sama. Inferensi di laptop tetap memakai fp16; adaptor yang dilatih dengan bf16 dapat
dimuat seperti biasa.

**Sebelum menghentikan instance:** pastikan kedua ZIP sudah terunduh.